In [ ]:


import os
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix



INPUT_DIR = "."
OUTPUT_DIR = "./feature_engineered"

os.makedirs(OUTPUT_DIR, exist_ok=True)


movies = pd.read_csv(
    os.path.join(INPUT_DIR, "4 movies_processed.csv")
)

reviews = pd.read_csv(
    os.path.join(INPUT_DIR, "5 reviews_processed.csv")
)

cast = pd.read_csv(
    os.path.join(INPUT_DIR, "1 cast_processed.csv")
)

crew = pd.read_csv(
    os.path.join(INPUT_DIR, "2 crew_processed.csv")
)

genres = pd.read_csv(
    os.path.join(INPUT_DIR, "3 genres_processed.csv")
)


print("=== UKURAN DATA ===")
print("Movies :", movies.shape)
print("Reviews:", reviews.shape)
print("Cast   :", cast.shape)
print("Crew   :", crew.shape)
print("Genres :", genres.shape)

print("\n=== KOLOM MOVIES ===")
print(movies.columns.tolist())

print("\n=== KOLOM REVIEWS ===")
print(reviews.columns.tolist())



def create_text_features(df, text_col):
    """
    Membuat fitur numerik dari teks:
    - jumlah karakter
    - jumlah kata
    - jumlah kalimat
    - rata-rata panjang kata
    - jumlah tanda seru
    - jumlah tanda tanya
    - rasio kata unik
    """

    text = df[text_col].fillna("").astype(str)

    df[f"{text_col}_char_count"] = text.str.len()


    df[f"{text_col}_word_count"] = text.str.split().str.len()

    df[f"{text_col}_sentence_count"] = (
        text.str.count(r"[.!?]")
    )

    df[f"{text_col}_avg_word_length"] = (
        text.apply(
            lambda x: np.mean(
                [len(word) for word in x.split()]
            ) if x.split() else 0
        )
    )

    df[f"{text_col}_exclamation_count"] = (
        text.str.count("!")
    )

    df[f"{text_col}_question_count"] = (
        text.str.count(r"\?")
    )

    def unique_word_ratio(x):
        words = x.split()

        if len(words) == 0:
            return 0

        return len(set(words)) / len(words)

    df[f"{text_col}_unique_word_ratio"] = (
        text.apply(unique_word_ratio)
    )

    return df



movies = create_text_features(
    movies,
    "overview_processed"
)

print("\n=== FITUR MOVIES ===")

movie_feature_cols = [
    "overview_processed_char_count",
    "overview_processed_word_count",
    "overview_processed_sentence_count",
    "overview_processed_avg_word_length",
    "overview_processed_exclamation_count",
    "overview_processed_question_count",
    "overview_processed_unique_word_ratio"
]

print(movie_feature_cols)

display(
    movies[movie_feature_cols].head()
)


reviews = create_text_features(
    reviews,
    "content_processed"
)

print("\n=== FITUR REVIEWS ===")

review_feature_cols = [
    "content_processed_char_count",
    "content_processed_word_count",
    "content_processed_sentence_count",
    "content_processed_avg_word_length",
    "content_processed_exclamation_count",
    "content_processed_question_count",
    "content_processed_unique_word_ratio"
]

print(review_feature_cols)

display(
    reviews[review_feature_cols].head()
)



print("\n=== MISSING VALUE MOVIES ===")
print(
    movies[movie_feature_cols]
    .isna()
    .sum()
)

print("\n=== MISSING VALUE REVIEWS ===")
print(
    reviews[review_feature_cols]
    .isna()
    .sum()
)



movies["overview_processed"] = (
    movies["overview_processed"]
    .fillna("")
    .astype(str)
)

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(
    movies["overview_processed"]
)

print("\n=== TF-IDF ===")
print("Jumlah dokumen :", X_tfidf.shape[0])
print("Jumlah fitur   :", X_tfidf.shape[1])



X_numeric = movies[movie_feature_cols].fillna(0)

scaler = StandardScaler()

X_numeric_scaled = scaler.fit_transform(
    X_numeric
)

X_numeric_scaled = csr_matrix(
    X_numeric_scaled
)

print("\n=== NUMERIC FEATURES ===")
print("Shape:", X_numeric_scaled.shape)



X_final = hstack([
    X_tfidf,
    X_numeric_scaled
])

print("\n=== FINAL FEATURE MATRIX ===")
print("Shape:", X_final.shape)



tfidf_feature_names = tfidf.get_feature_names_out()

final_feature_names = list(tfidf_feature_names) + movie_feature_cols

print("\nJumlah nama fitur:", len(final_feature_names))



movies_output = os.path.join(
    OUTPUT_DIR,
    "movies_feature_engineered.csv"
)

reviews_output = os.path.join(
    OUTPUT_DIR,
    "reviews_feature_engineered.csv"
)

movies.to_csv(
    movies_output,
    index=False
)

reviews.to_csv(
    reviews_output,
    index=False
)

print("\n=== FILE BERHASIL DISIMPAN ===")
print(movies_output)
print(reviews_output)


print("\n====================================")
print("FEATURE ENGINEERING SELESAI")
print("====================================")

print("Movies awal          :", movies.shape)
print("Reviews awal         :", reviews.shape)
print("TF-IDF features      :", X_tfidf.shape[1])
print("Numeric features     :", len(movie_feature_cols))
print("Total final features :", X_final.shape[1])
print("Output directory     :", OUTPUT_DIR)

=== UKURAN DATA ===
Movies : (220, 14)
Reviews: (697, 7)
Cast   : (2923, 6)
Crew   : (606, 5)
Genres : (557, 3)

=== KOLOM MOVIES ===
['tmdb_id', 'title', 'release_date', 'original_language', 'runtime', 'budget', 'revenue', 'vote_average', 'vote_count', 'popularity', 'overview', 'title_clean', 'overview_clean', 'overview_processed']

=== KOLOM REVIEWS ===
['tmdb_id', 'author', 'rating', 'content', 'author_clean', 'content_clean', 'content_processed']

=== FITUR MOVIES ===
['overview_processed_char_count', 'overview_processed_word_count', 'overview_processed_sentence_count', 'overview_processed_avg_word_length', 'overview_processed_exclamation_count', 'overview_processed_question_count', 'overview_processed_unique_word_ratio']


,overview_processed_char_count,overview_processed_word_count,overview_processed_sentence_count,overview_processed_avg_word_length,overview_processed_exclamation_count,overview_processed_question_count,overview_processed_unique_word_ratio
0,210,32,0,5.593750,0,0,1.000000
1,120,18,0,5.722222,0,0,0.944444
2,196,26,0,6.576923,0,0,1.000000
3,228,29,0,6.896552,0,0,1.000000
4,132,18,0,6.388889,0,0,0.944444



=== FITUR REVIEWS ===
['content_processed_char_count', 'content_processed_word_count', 'content_processed_sentence_count', 'content_processed_avg_word_length', 'content_processed_exclamation_count', 'content_processed_question_count', 'content_processed_unique_word_ratio']


,content_processed_char_count,content_processed_word_count,content_processed_sentence_count,content_processed_avg_word_length,content_processed_exclamation_count,content_processed_question_count,content_processed_unique_word_ratio
0,554,81,0,5.851852,0,0,0.962963
1,418,62,0,5.758065,0,0,0.790323
2,620,88,0,6.056818,0,0,0.795455
3,2294,294,0,6.806122,0,0,0.860544
4,783,130,0,5.030769,0,0,0.746154



=== MISSING VALUE MOVIES ===
overview_processed_char_count           0
overview_processed_word_count           0
overview_processed_sentence_count       0
overview_processed_avg_word_length      0
overview_processed_exclamation_count    0
overview_processed_question_count       0
overview_processed_unique_word_ratio    0
dtype: int64

=== MISSING VALUE REVIEWS ===
content_processed_char_count           0
content_processed_word_count           0
content_processed_sentence_count       0
content_processed_avg_word_length      0
content_processed_exclamation_count    0
content_processed_question_count       0
content_processed_unique_word_ratio    0
dtype: int64

=== TF-IDF ===
Jumlah dokumen : 220
Jumlah fitur   : 1017

=== NUMERIC FEATURES ===
Shape: (220, 7)

=== FINAL FEATURE MATRIX ===
Shape: (220, 1024)

Jumlah nama fitur: 1024

=== FILE BERHASIL DISIMPAN ===
./feature_engineered/movies_feature_engineered.csv
./feature_engineered/reviews_feature_engineered.csv

FEATURE ENGINEERING S

## 6. Feature Engineering dan Feature Extraction

Tahap ini menambahkan fitur numerik dari teks dan melakukan ekstraksi fitur menggunakan TF-IDF.

In [ ]:


import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack


def create_text_features(df, text_col):
    """Membuat fitur numerik dari kolom teks."""
    text = df[text_col].fillna("").astype(str)

    df[f"{text_col}_char_count"] = text.str.len()
    df[f"{text_col}_word_count"] = text.str.split().str.len()
    df[f"{text_col}_sentence_count"] = text.str.count(r"[.!?]")

    df[f"{text_col}_avg_word_length"] = text.apply(
        lambda x: np.mean([len(w) for w in x.split()])
        if len(x.split()) > 0 else 0
    )

    df[f"{text_col}_exclamation_count"] = text.str.count("!")
    df[f"{text_col}_question_count"] = text.str.count(r"\?")

    df[f"{text_col}_unique_word_ratio"] = text.apply(
        lambda x: (
            len(set(x.split())) / len(x.split())
            if len(x.split()) > 0 else 0
        )
    )

    return df

movies = create_text_features(
    movies,
    "overview_processed"
)

print("Feature engineering movies selesai.")
print(movies.head())

reviews = create_text_features(
    reviews,
    "content_processed"
)

print("\nFeature engineering reviews selesai.")
print(reviews.head())


tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(
    movies["overview_processed"].fillna("")
)

print("\nTF-IDF selesai.")
print("Ukuran matriks TF-IDF :", X_tfidf.shape)
print("Jumlah fitur TF-IDF  :", len(tfidf.get_feature_names_out()))

numeric_features = [
    "overview_processed_char_count",
    "overview_processed_word_count",
    "overview_processed_sentence_count",
    "overview_processed_avg_word_length",
    "overview_processed_exclamation_count",
    "overview_processed_question_count",
    "overview_processed_unique_word_ratio"
]

X_numeric = movies[numeric_features].fillna(0)

scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)

X_final = hstack([
    X_tfidf,
    X_numeric_scaled
])

print("\nFeature engineering + feature extraction selesai.")
print("Shape fitur numerik :", X_numeric_scaled.shape)
print("Shape fitur TF-IDF  :", X_tfidf.shape)
print("Shape fitur akhir   :", X_final.shape)

movies.to_csv(
    f"{OUTPUT_DIR}/movies_feature_engineered.csv",
    index=False
)

reviews.to_csv(
    f"{OUTPUT_DIR}/reviews_feature_engineered.csv",
    index=False
)

print("\nHasil feature engineering tersimpan di folder processed/.")


Feature engineering movies selesai.
   tmdb_id                      title release_date original_language  runtime  \
0   969681  Spider-Man: Brand New Day   2026-07-29                en      145   
1  1204680            Coyote vs. Acme   2026-08-20                en      103   
2  1368337                The Odyssey   2026-07-15                en      173   
3  1375646                     Colony   2026-05-21                ko      123   
4  1288445                     Mutiny   2026-08-19                en       96   

      budget     revenue  vote_average  vote_count  popularity  ...  \
0  225000000  2478502286         7.861        2873    630.0457  ...   
1   72000000    79825108         7.558         518    335.4296  ...   
2  250000000  1723918419         8.017        3903    333.2210  ...   
3          0           0         8.071         876    243.0215  ...   
4   40000000    20824283         6.743         498    209.3601  ...   

                 title_clean  \
0  Spider-Man: Bra